In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import ast
import json
import os
import re
import time
import subprocess
import zipfile

DATASET_DIR = '/content/drive/MyDrive/My Research/APR Research/Dataset'

KB_PATH = f'{DATASET_DIR}/pattern_knowledge_base_final.json'
TEST_PATH = f'{DATASET_DIR}/test_loop_filtered_verified.jsonl'
ZIP_PATH = f'{DATASET_DIR}/public_test_cases.zip'
EXTRACT_DIR = f'{DATASET_DIR}/test_cases'
TEST_CASES_DIR = f'{EXTRACT_DIR}/codenet/public_test_cases'

OUTPUT_PATH = f'{DATASET_DIR}/phase2_batch_results_v2.jsonl'

SPEEDUP_THRESHOLD = 1.10
MAX_ATTEMPTS = 5

# Load Knowledge Base
with open(KB_PATH) as f:
    pattern_knowledge_base = json.load(f)
print(f'Loaded Knowledge Base: {len(pattern_knowledge_base)} patterns')

# Load test samples
test_samples = []
with open(TEST_PATH) as f:
    for line in f:
        test_samples.append(json.loads(line))
print(f'Loaded {len(test_samples)} test samples')

# Extract test cases zip if not already extracted
if not os.path.isdir(TEST_CASES_DIR):
    print('Extracting test cases zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
print(f'Test cases available for {len(os.listdir(TEST_CASES_DIR))} problems')

In [ ]:
!pip install -U bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

GEN_MODEL_NAME = "deepseek-ai/deepseek-coder-6.7b-instruct"

gen_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    quantization_config=gen_bnb_config,
    device_map="auto",
)


def call_gen_llm(prompt, max_new_tokens=800):
    messages = [{"role": "user", "content": prompt}]
    inputs = gen_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(gen_model.device)

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    input_length = inputs['input_ids'].shape[1]
    return gen_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

print('DeepSeek-Coder-6.7B-Instruct loaded.')

In [ ]:
def extract_code_from_response(response):
    """
    Robustly extracts Python code from an LLM response that may have
    prose before/after a ```python ... ``` fence, or no fence at all.
    """
    match = re.search(r"```(?:python)?\s*\n(.*?)```", response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()

In [ ]:
def load_test_cases(problem_id, test_cases_dir):
    problem_dir = os.path.join(test_cases_dir, problem_id)
    if not os.path.isdir(problem_dir):
        return []

    cases = []
    i = 0
    while True:
        input_path = os.path.join(problem_dir, f'input.{i}.txt')
        output_path = os.path.join(problem_dir, f'output.{i}.txt')
        if not os.path.exists(input_path):
            break
        with open(input_path) as f:
            stdin_data = f.read()
        with open(output_path) as f:
            expected_output = f.read().strip()
        cases.append({'stdin': stdin_data, 'expected': expected_output})
        i += 1
    return cases


def check_compiles(code):
    try:
        compile(code, '<repair>', 'exec')
        return True, None
    except SyntaxError as e:
        return False, str(e)


def run_with_timeout(code, stdin_data, timeout_sec=5):
    start = time.time()
    try:
        result = subprocess.run(
            ['python3', '-c', code],
            input=stdin_data,
            capture_output=True,
            text=True,
            timeout=timeout_sec,
        )
        elapsed = time.time() - start
        return result.stdout.strip(), result.returncode == 0, elapsed
    except subprocess.TimeoutExpired:
        return None, False, timeout_sec


def check_functional_correctness(code, test_cases):
    for case in test_cases:
        output, success, _ = run_with_timeout(code, case['stdin'])
        if not success or output != case['expected']:
            return False, {
                **case,
                'actual_output': output if output is not None else '(crashed or timed out)',
            }
    return True, None


def measure_runtime(code, test_cases, repeats=3):
    total_time = 0
    for _ in range(repeats):
        for case in test_cases:
            _, success, elapsed = run_with_timeout(code, case['stdin'])
            if not success:
                return None
            total_time += elapsed
    return total_time / repeats


def validate_repair(slow_code, repaired_code, problem_id, test_cases_dir, speedup_threshold=SPEEDUP_THRESHOLD):
    test_cases = load_test_cases(problem_id, test_cases_dir)
    if not test_cases:
        return {'verdict': 'RETRY', 'reason': f'No test cases found for {problem_id}'}

    compiles, syntax_error = check_compiles(repaired_code)
    if not compiles:
        return {'verdict': 'RETRY', 'reason': 'compile_error', 'detail': syntax_error}

    correct, failing_case = check_functional_correctness(repaired_code, test_cases)
    if not correct:
        return {'verdict': 'RETRY', 'reason': 'test_failure', 'detail': failing_case}

    slow_runtime = measure_runtime(slow_code, test_cases)
    repaired_runtime = measure_runtime(repaired_code, test_cases)

    if slow_runtime is None or repaired_runtime is None or repaired_runtime <= 0:
        return {'verdict': 'RETRY', 'reason': 'runtime_measurement_failed'}

    speedup = slow_runtime / repaired_runtime

    if speedup >= speedup_threshold:
        return {
            'verdict': 'ACCEPT',
            'speedup': round(speedup, 3),
            'slow_runtime': round(slow_runtime, 4),
            'repaired_runtime': round(repaired_runtime, 4),
        }
    else:
        return {
            'verdict': 'RETRY',
            'reason': 'insufficient_speedup',
            'speedup': round(speedup, 3),
            'slow_runtime': round(slow_runtime, 4),
            'repaired_runtime': round(repaired_runtime, 4),
        }

print('Validation ready.')

In [ ]:
def load_completed_problem_ids(output_path):
    completed = set()
    if os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    completed.add(rec['problem_id'])
                except Exception:
                    continue
    return completed

In [ ]:
def build_prompting_baseline_prompt(slow_code):
    return f"""You are an expert Python engineer specializing in performance optimization.

Optimize the following Python code to run faster while preserving its exact behavior.

Slow code:
{slow_code}

Return ONLY the optimized Python code. No markdown. No explanations."""


def build_prompting_feedback_prompt(slow_code, previous_code, validation_result):
    reason = validation_result.get('reason')
    if reason == 'compile_error':
        feedback = f"Your previous attempt had a syntax error: {validation_result.get('detail')}"
    elif reason == 'test_failure':
        case = validation_result.get('detail', {})
        feedback = (
            f"Your previous attempt failed a test case.\n"
            f"Input:\n{case.get('stdin', '')}\n"
            f"Expected output:\n{case.get('expected', '')}\n"
            f"Your code's ACTUAL output:\n{case.get('actual_output', '')}\n"
            f"Fix the logic error."
        )
    elif reason == 'insufficient_speedup':
        feedback = (
            f"Your previous attempt was correct but NOT fast enough "
            f"({validation_result.get('speedup')}x, need >= {SPEEDUP_THRESHOLD}x). "
            f"Try a genuinely more efficient approach."
        )
    else:
        feedback = "Your previous attempt did not pass validation. Try a different approach."

    base_prompt = build_prompting_baseline_prompt(slow_code)
    return f"""{base_prompt}

Your previous attempt:
{previous_code}

Feedback on that attempt:
{feedback}

Generate a NEW, corrected version that addresses this feedback."""


def prompting_repair_loop(slow_code, problem_id, test_cases_dir, max_attempts=MAX_ATTEMPTS):
    prompt = build_prompting_baseline_prompt(slow_code)
    current_code = extract_code_from_response(call_gen_llm(prompt, max_new_tokens=800))
    history = []

    for attempt in range(1, max_attempts + 1):
        result = validate_repair(slow_code, current_code, problem_id, test_cases_dir)
        result['attempt'] = attempt
        history.append(result)

        if result['verdict'] == 'ACCEPT':
            return {'final_verdict': 'ACCEPT', 'code': current_code, 'history': history}
        if attempt == max_attempts:
            return {'final_verdict': 'RETRY_EXHAUSTED', 'code': current_code, 'history': history}

        fb_prompt = build_prompting_feedback_prompt(slow_code, current_code, result)
        current_code = extract_code_from_response(call_gen_llm(fb_prompt, max_new_tokens=800))

    return {'final_verdict': 'RETRY_EXHAUSTED', 'code': current_code, 'history': history}


def run_prompting_baseline(test_samples, test_cases_dir, output_path, max_attempts=MAX_ATTEMPTS):
    completed = load_completed_problem_ids(output_path)
    print(f'Already completed: {len(completed)} / {len(test_samples)}')

    with open(output_path, 'a') as out_f:
        for i, sample in enumerate(test_samples):
            problem_id = sample.get('problem_id')
            slow_code = sample['input']
            if problem_id in completed:
                continue

            print(f'[{i+1}/{len(test_samples)}] {problem_id}...')
            start = time.time()
            try:
                loop_result = prompting_repair_loop(slow_code, problem_id, test_cases_dir, max_attempts)
                record = {
                    'problem_id': problem_id,
                    'final_verdict': loop_result['final_verdict'],
                    'num_attempts': len(loop_result['history']),
                    'history': loop_result['history'],
                    'final_code': loop_result['code'],
                    'elapsed_seconds': round(time.time() - start, 2),
                }
            except Exception as e:
                record = {'problem_id': problem_id, 'final_verdict': 'PIPELINE_ERROR', 'error': str(e)}

            out_f.write(json.dumps(record) + '\n')
            out_f.flush()

    print('Prompting baseline run complete.')


In [ ]:
PROMPTING_OUTPUT_PATH = f'{DATASET_DIR}/prompting_baseline_results.jsonl'
run_prompting_baseline(test_samples, TEST_CASES_DIR, PROMPTING_OUTPUT_PATH, max_attempts=MAX_ATTEMPTS)

In [ ]:
import statistics

def score_baseline_results(output_path, label):
    rows = []
    with open(output_path) as f:
        for line in f:
            rows.append(json.loads(line))

    total = len(rows)
    accepted = [r for r in rows if r.get('final_verdict') == 'ACCEPT']
    pipeline_errors = [r for r in rows if r.get('final_verdict') in ('PIPELINE_ERROR', 'PARSE_ERROR')]

    pass1_count = 0
    for r in rows:
        history = r.get('history', [])
        if not history:
            continue
        first = history[0]
        if first.get('verdict') == 'ACCEPT' or first.get('reason') == 'insufficient_speedup':
            pass1_count += 1

    print(f'=== {label} ===')
    print(f'Total: {total}')
    print(f'%OPT (accept rate): {len(accepted)/total*100:.1f}%')
    print(f'Pass@1: {pass1_count/total*100:.1f}%')
    print(f'Pipeline/parse errors: {len(pipeline_errors)}')

    if accepted:
        speedups = [r['history'][-1]['speedup'] for r in accepted]
        print(f'Median speedup: {statistics.median(speedups):.3f}x')
        print(f'Mean speedup: {statistics.mean(speedups):.3f}x')
    print()

score_baseline_results(PROMPTING_OUTPUT_PATH, 'Prompting Baseline')